# Multimodal Sentiment Analysis

## Phân Tích Ý Kiến Người Dùng Qua Văn Bản và Hình Ảnh

**Kiến trúc hệ thống:**
- **Vision Encoder**: Google ViT-L/16 (Vision Transformer)
- **Projection Layer**: MLP Bridge (1024 → 4096)
- **LLM**: InternVL3-8B-Instruct

```
                      [ Embeddings ]───( Word-pieces )───>[ de-Tokenizer ]
                            ^                                    |
                            |                                    v
                      ( Lang tokens )                      (( Output Text ))
                            ^
                            |
    +-------------------------------------------------------+
    |                         LLM                           |
    +-------------------------------------------------------+
         ^                               ^
         |                               |
   (Lang tokens)                   (Lang tokens)
         |                               |
    [ Projection ]                [  Embeddings  ]
         ^                               ^
    (Img tokens)                   (word-pieces)
         |                               |
    [    ViT     ]                [  Tokenizer   ]
         ^                               ^
   (14px patches)                        |
         |                               |
    [Preprocessor]                       |
         ^                               |
         |                               |
  ((   Image   ))                 (( Input text  ))
```

**Data Flow:**
- **Image Path**: Image → Preprocessor (512→224) → ViT (197 patches × 1024) → Projection (4096) → LLM
- **Text Path**: Input Text → Tokenizer → Embeddings (4096) → LLM
- **Output**: LLM → de-Tokenizer → Generated Text (Sentiment Analysis)

## 1. Cai Dat Dependencies

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install torch torchvision transformers accelerate pillow einops timm gradio sentencepiece -q

In [ ]:
# Import các thư viện
import torch
import torch.nn as nn
from PIL import Image
from transformers import ViTModel, ViTImageProcessor, AutoModel, AutoTokenizer
from dataclasses import dataclass
from typing import Optional, List
import requests
from io import BytesIO

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

---
## 2. Preprocessor va ViT (Vision Encoder)

**Preprocessor**: Chuyen doi anh sang tensor, resize ve 224x224
**ViT (Vision Transformer)**: Chia anh thanh cac patches 16x16 va xu ly nhu sequence tokens.

| Thong so | Gia tri |
|----------|--------|
| Model | `google/vit-large-patch16-224` |
| Input | 224x224 |
| Patch Size | 16x16 |
| Hidden Dim | 1024 |
| Num Tokens | 197 (196 patches + 1 CLS) |

In [ ]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

# ============================================
# PREPROCESSOR - Chuyen doi anh sang tensor
# ============================================

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size=224):
    """
    Preprocessor: Image -> Tensor
    
    Input: PIL Image (any size)
    Output: Tensor [3, input_size, input_size] normalized
    """
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

# ============================================
# VIT - Vision Encoder
# ============================================

class VisionEncoder(nn.Module):
    """
    ViT: Image patches -> Img tokens
    
    Input: Tensor [B, 3, 224, 224]
    Output: Img tokens [B, 197, 1024]
    """
    
    def __init__(
        self, 
        model_name: str = "google/vit-large-patch16-224",
        device: str = "cuda",
        torch_dtype: torch.dtype = torch.float16
    ):
        super().__init__()
        self.device = device
        self.torch_dtype = torch_dtype
        
        print(f"Loading ViT: {model_name}")
        self.model = ViTModel.from_pretrained(
            model_name,
            torch_dtype=torch_dtype
        ).to(device)
        
        self.processor = ViTImageProcessor.from_pretrained(model_name)
        
        # Freeze weights
        for param in self.model.parameters():
            param.requires_grad = False
        self.model.eval()
        
        self.hidden_size = self.model.config.hidden_size  # 1024
        self.num_patches = (self.model.config.image_size // self.model.config.patch_size) ** 2 + 1
        
        print(f"Loaded: hidden_size={self.hidden_size}, num_patches={self.num_patches}")
    
    @torch.no_grad()
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Extract visual features.
        
        Input: [B, 3, 224, 224]
        Output: [B, 197, 1024] - Img tokens
        """
        pixel_values = pixel_values.to(self.device, dtype=self.torch_dtype)
        outputs = self.model(pixel_values=pixel_values)
        return outputs.last_hidden_state  # [B, 197, 1024]

### Test Preprocessor va ViT

In [ ]:
# Test Preprocessor va ViT
print("=== Testing Preprocessor + ViT ===")

# Download test image
url = "https://images.unsplash.com/photo-1575936123452-b67c3203c357?w=400"
response = requests.get(url)
test_image = Image.open(BytesIO(response.content)).convert("RGB")
print(f"Original image size: {test_image.size}")
display(test_image.resize((224, 224)))

# Preprocessor: Image -> Tensor
transform = build_transform(input_size=224)
pixel_values = transform(test_image).unsqueeze(0)  # [1, 3, 224, 224]
print(f"After Preprocessor: {pixel_values.shape}")

# ViT: Tensor -> Img tokens
vision_encoder = VisionEncoder(device=device)
img_tokens = vision_encoder(pixel_values)
print(f"After ViT (Img tokens): {img_tokens.shape}")  # [1, 197, 1024]

---
## 3. Projection Layer (MLP)

Chuyen doi Img tokens tu ViT (1024 dim) sang Lang tokens cho LLM (4096 dim).

```
Img tokens [B, 197, 1024] -> Linear(1024, 4096) -> GELU -> Linear(4096, 4096) -> Lang tokens [B, 197, 4096]
```

In [ ]:
class MLPProjector(nn.Module):
    """
    Projection: Img tokens -> Lang tokens
    
    Input: [B, 197, 1024] tu ViT
    Output: [B, 197, 4096] cho LLM
    """
    
    def __init__(
        self,
        vision_dim: int = 1024,   # ViT hidden dim
        llm_dim: int = 4096,      # LLM hidden dim
    ):
        super().__init__()
        
        self.projector = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )
        
        # Initialize weights
        for module in self.projector:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def forward(self, img_tokens: torch.Tensor) -> torch.Tensor:
        """
        Project Img tokens to Lang tokens.
        
        Input: [B, 197, 1024]
        Output: [B, 197, 4096]
        """
        return self.projector(img_tokens)

In [ ]:
# Test Projection Layer
print("=== Testing Projection Layer ===")

projector = MLPProjector(vision_dim=1024, llm_dim=4096).to(device).half()
print(f"Parameters: {sum(p.numel() for p in projector.parameters()):,}")

# Project: Img tokens -> Lang tokens
lang_tokens_from_img = projector(img_tokens)
print(f"Input (Img tokens): {img_tokens.shape}")
print(f"Output (Lang tokens): {lang_tokens_from_img.shape}")

---
## 4. LLM - InternVL3-8B-Instruct

LLM nhan:
- Lang tokens tu Projection (Image path)
- Lang tokens tu Embeddings (Text path)

Va output text qua de-Tokenizer.

| Model | Parameters | VRAM |
|-------|------------|------|
| InternVL3-1B | 1B | ~4GB |
| InternVL3-2B | 2B | ~6GB |
| InternVL3-8B-Instruct | 8B | ~18GB |

In [ ]:
# ============================================
# LOAD LLM - InternVL3-8B
# ============================================

MODEL_NAME = "OpenGVLab/InternVL3-8B-Instruct"

print(f"Loading {MODEL_NAME}...")
print("Lan dau se download ~16GB, vui long cho...")

# Determine dtype
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

# Load tokenizer (Text -> word-pieces -> Embeddings -> Lang tokens)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Load model (LLM + de-Tokenizer)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map="auto"
).eval()

print("Model loaded!")

---
## 5. Dataset va DataLoader

In [ ]:
import json
import os
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ============================================
# LABEL DEFINITIONS - Chi co 3 Sentiments
# ============================================

SENTIMENT_LABELS = ["Positive", "Negative", "Neutral"]

# Tao mapping label -> id
LABEL2ID = {label: idx for idx, label in enumerate(SENTIMENT_LABELS)}
ID2LABEL = {idx: label for idx, label in enumerate(SENTIMENT_LABELS)}
NUM_LABELS = len(LABEL2ID)

print(f"Number of labels: {NUM_LABELS}")
print(f"Labels: {list(LABEL2ID.keys())}")

In [ ]:
# ============================================
# DATA LOADER - Doc train/dev/test.json
# ============================================

def load_dataset_json(json_path: str) -> list:
    """
    Doc file JSON dataset.
    
    Args:
        json_path: Duong dan toi file JSON
        
    Returns:
        List cac samples
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} samples from {json_path}")
    return data


def load_all_splits(data_dir: str = "datasets"):
    """
    Load tat ca cac split: train, dev, test.
    
    Args:
        data_dir: Thu muc chua data
        
    Returns:
        Dict chua train, dev, test data
    """
    splits = {}
    for split in ["train", "dev", "test"]:
        json_path = os.path.join(data_dir, f"{split}.json")
        if os.path.exists(json_path):
            splits[split] = load_dataset_json(json_path)
        else:
            print(f"[WARNING] {json_path} not found!")
            splits[split] = []
    return splits


# Load data
DATA_DIR = "datasets"
IMAGE_DIR = os.path.join(DATA_DIR, "image")

dataset_splits = load_all_splits(DATA_DIR)

print(f"\nDataset Statistics:")
print(f"  Train: {len(dataset_splits['train'])} samples")
print(f"  Dev:   {len(dataset_splits['dev'])} samples")
print(f"  Test:  {len(dataset_splits['test'])} samples")

In [ ]:
# ============================================
# PYTORCH DATASET CLASS
# ============================================

class SentimentDataset(Dataset):
    """
    PyTorch Dataset cho Sentiment Analysis.
    
    Moi sample gom:
    - comment: Text binh luan
    - list_img: Danh sach ten file anh
    - labels: Sentiment labels (Positive/Negative/Neutral)
    """
    
    def __init__(
        self,
        data: list,
        image_dir: str,
        label2id: dict,
        transform=None,
    ):
        self.data = data
        self.image_dir = image_dir
        self.label2id = label2id
        self.num_labels = len(label2id)
        self.transform = transform if transform else build_transform(224)
        
        self.samples = self._prepare_samples()
        print(f"Prepared {len(self.samples)} valid samples")
    
    def _extract_sentiment(self, label: str) -> str:
        """Extract sentiment tu label goc (VD: 'Room#Positive' -> 'Positive')."""
        if "#" in label:
            return label.split("#")[1]
        return label
    
    def _prepare_samples(self) -> list:
        """Loc va chuan bi samples hop le."""
        valid_samples = []
        
        for item in self.data:
            if not item.get("list_img") or len(item["list_img"]) == 0:
                continue
            
            raw_labels = item.get("text_img_label", [])
            if not raw_labels:
                continue
            
            # Extract sentiments tu labels goc
            sentiments = set()
            for label in raw_labels:
                sentiment = self._extract_sentiment(label)
                if sentiment in self.label2id:
                    sentiments.add(sentiment)
            
            if not sentiments:
                continue
            
            img_name = item["list_img"][0]
            img_path = os.path.join(self.image_dir, img_name)
            if not os.path.exists(img_path):
                continue
            
            valid_samples.append({
                "comment": item.get("comment", ""),
                "image_path": img_path,
                "labels": list(sentiments),
            })
        
        return valid_samples
    
    def __len__(self):
        return len(self.samples)
    
    def _labels_to_tensor(self, labels: list) -> torch.Tensor:
        """Convert list labels thanh multi-hot tensor."""
        label_tensor = torch.zeros(self.num_labels)
        for label in labels:
            if label in self.label2id:
                label_tensor[self.label2id[label]] = 1.0
        return label_tensor
    
    def __getitem__(self, idx: int) -> dict:
        sample = self.samples[idx]
        
        # Load image
        image = Image.open(sample["image_path"]).convert("RGB")
        
        # Preprocessor: Image -> Tensor
        pixel_values = self.transform(image)
        
        # Convert labels to tensor
        label_tensor = self._labels_to_tensor(sample["labels"])
        
        return {
            "pixel_values": pixel_values,
            "labels": label_tensor,
            "comment": sample["comment"],
            "image_path": sample["image_path"],
        }


def collate_fn(batch):
    """Custom collate function cho DataLoader."""
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])
    comments = [item["comment"] for item in batch]
    image_paths = [item["image_path"] for item in batch]
    
    return {
        "pixel_values": pixel_values,
        "labels": labels,
        "comments": comments,
        "image_paths": image_paths,
    }

In [ ]:
# ============================================
# TAO DATALOADERS
# ============================================

# Tao datasets
train_dataset = SentimentDataset(
    data=dataset_splits["train"],
    image_dir=IMAGE_DIR,
    label2id=LABEL2ID,
)

dev_dataset = SentimentDataset(
    data=dataset_splits["dev"],
    image_dir=IMAGE_DIR,
    label2id=LABEL2ID,
)

test_dataset = SentimentDataset(
    data=dataset_splits["test"],
    image_dir=IMAGE_DIR,
    label2id=LABEL2ID,
)

# Tao dataloaders
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

print(f"\nDataLoader Statistics:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Dev batches:   {len(dev_loader)}")
print(f"  Test batches:  {len(test_loader)}")

# Test 1 batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  pixel_values shape: {sample_batch['pixel_values'].shape}")
print(f"  labels shape: {sample_batch['labels'].shape}")

---
## 6. Training Loop

Fine-tune Projection Layer, giu nguyen (freeze) ViT.

In [ ]:
# ============================================
# MULTIMODAL SENTIMENT MODEL
# ============================================
# Architecture theo diagram:
# Image -> Preprocessor -> ViT -> Projection -> LLM -> de-Tokenizer -> Output Text
# Text  -> Tokenizer -> Embeddings -> LLM -> de-Tokenizer -> Output Text

class MultimodalSentimentModel(nn.Module):
    """
    Model phan loai sentiment theo kien truc VLM.
    
    Architecture:
    - Preprocessor: Image -> Tensor [B, 3, 224, 224]
    - ViT (frozen): Tensor -> Img tokens [B, 197, 1024]
    - Projection (trainable): Img tokens -> Lang tokens [B, 197, 4096]
    - Classifier Head: Lang tokens -> Logits [B, num_labels]
    
    Note: Trong training, ta thay LLM bang Classifier Head de train nhanh hon.
    Khi inference voi LLM, ta dung truc tiep model.chat() cua InternVL3.
    """
    
    def __init__(
        self,
        vision_encoder: VisionEncoder,
        projector: MLPProjector,
        num_labels: int,
        dropout: float = 0.3,
    ):
        super().__init__()
        
        self.vision_encoder = vision_encoder
        self.projector = projector
        self.num_labels = num_labels
        
        # Freeze vision encoder
        for param in self.vision_encoder.parameters():
            param.requires_grad = False
        
        # Classifier head (thay the LLM trong training)
        # Mean pool Lang tokens -> single vector -> classify
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(4096, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, num_labels),
        )
        
        # Initialize classifier
        for module in self.classifier:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            pixel_values: [B, 3, 224, 224]
            
        Returns:
            logits: [B, num_labels]
        """
        # ViT: Image -> Img tokens (frozen)
        with torch.no_grad():
            img_tokens = self.vision_encoder(pixel_values)  # [B, 197, 1024]
        
        # Projection: Img tokens -> Lang tokens (trainable)
        lang_tokens = self.projector(img_tokens)  # [B, 197, 4096]
        
        # Mean pooling over tokens
        pooled = lang_tokens.mean(dim=1)  # [B, 4096]
        
        # Classify
        logits = self.classifier(pooled)  # [B, num_labels]
        
        return logits
    
    def get_trainable_params(self):
        """Tra ve so luong trainable parameters."""
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


# Tao model
sentiment_model = MultimodalSentimentModel(
    vision_encoder=vision_encoder,
    projector=projector,
    num_labels=NUM_LABELS,
).to(device)

total_params, trainable_params = sentiment_model.get_trainable_params()
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
# ============================================
# TRAINING CONFIGURATION
# ============================================

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Hyperparameters
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10

# Optimizer - chi train projection layer va classifier
trainable_params_list = [p for p in sentiment_model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params_list, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Scheduler
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)

# Loss function - Multi-label classification
criterion = nn.BCEWithLogitsLoss()

print(f"Training Configuration:")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Total Steps: {total_steps}")

In [ ]:
# ============================================
# TRAINING LOOP
# ============================================

def train_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    """Train 1 epoch."""
    model.train()
    total_loss = 0
    num_batches = 0
    
    progress_bar = tqdm(dataloader, desc="Training")
    
    for batch in progress_bar:
        pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        logits = model(pixel_values)
        
        loss = criterion(logits.float(), labels.float())
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / num_batches
    return avg_loss


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    """Validate model."""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    all_preds = []
    all_labels = []
    
    for batch in tqdm(dataloader, desc="Validating"):
        pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
        labels = batch["labels"].to(device)
        
        logits = model(pixel_values)
        loss = criterion(logits.float(), labels.float())
        
        total_loss += loss.item()
        num_batches += 1
        
        preds = torch.sigmoid(logits) > 0.5
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
    
    avg_loss = total_loss / num_batches
    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    
    return avg_loss, all_preds, all_labels

In [ ]:
# ============================================
# RUN TRAINING
# ============================================

best_val_loss = float('inf')
train_losses = []
val_losses = []

print("Starting Training...")
print("=" * 50)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 30)
    
    train_loss = train_epoch(
        sentiment_model, train_loader, optimizer, scheduler, criterion, device
    )
    train_losses.append(train_loss)
    
    val_loss, val_preds, val_labels = validate(
        sentiment_model, dev_loader, criterion, device
    )
    val_losses.append(val_loss)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': sentiment_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, 'best_model.pt')
        print("[SAVED] Best model saved!")

print("\n" + "=" * 50)
print("Training Complete!")
print(f"Best Validation Loss: {best_val_loss:.4f}")

In [ ]:
# ============================================
# PLOT TRAINING CURVES
# ============================================

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

epochs_range = range(1, len(train_losses) + 1)
ax.plot(epochs_range, train_losses, 'b-', label='Train Loss')
ax.plot(epochs_range, val_losses, 'r-', label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

---
## 7. Evaluation (Accuracy, Precision, Recall, F1)

In [ ]:
# ============================================
# EVALUATION METRICS
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    hamming_loss,
)
import numpy as np


def compute_metrics(preds: torch.Tensor, labels: torch.Tensor, id2label: dict) -> dict:
    """
    Tinh cac metrics cho multi-label classification.
    
    Args:
        preds: Predictions tensor [N, num_labels]
        labels: Ground truth tensor [N, num_labels]
        id2label: Mapping id -> label name
        
    Returns:
        Dict chua cac metrics
    """
    preds_np = preds.numpy()
    labels_np = labels.numpy()
    
    # Overall metrics (micro, macro, weighted)
    metrics = {
        # Micro metrics - tinh tren tat ca predictions
        "micro_precision": precision_score(labels_np, preds_np, average='micro', zero_division=0),
        "micro_recall": recall_score(labels_np, preds_np, average='micro', zero_division=0),
        "micro_f1": f1_score(labels_np, preds_np, average='micro', zero_division=0),
        
        # Macro metrics - trung binh cua tung class
        "macro_precision": precision_score(labels_np, preds_np, average='macro', zero_division=0),
        "macro_recall": recall_score(labels_np, preds_np, average='macro', zero_division=0),
        "macro_f1": f1_score(labels_np, preds_np, average='macro', zero_division=0),
        
        # Weighted metrics
        "weighted_precision": precision_score(labels_np, preds_np, average='weighted', zero_division=0),
        "weighted_recall": recall_score(labels_np, preds_np, average='weighted', zero_division=0),
        "weighted_f1": f1_score(labels_np, preds_np, average='weighted', zero_division=0),
        
        # Samples metrics
        "samples_f1": f1_score(labels_np, preds_np, average='samples', zero_division=0),
        
        # Hamming loss
        "hamming_loss": hamming_loss(labels_np, preds_np),
        
        # Exact match ratio
        "exact_match_ratio": np.mean(np.all(preds_np == labels_np, axis=1)),
    }
    
    return metrics


def print_metrics(metrics: dict):
    """In metrics dep."""
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)
    
    print("\n[MICRO METRICS] (tren tat ca predictions)")
    print(f"  Precision: {metrics['micro_precision']:.4f}")
    print(f"  Recall:    {metrics['micro_recall']:.4f}")
    print(f"  F1-Score:  {metrics['micro_f1']:.4f}")
    
    print("\n[MACRO METRICS] (trung binh cua tung class)")
    print(f"  Precision: {metrics['macro_precision']:.4f}")
    print(f"  Recall:    {metrics['macro_recall']:.4f}")
    print(f"  F1-Score:  {metrics['macro_f1']:.4f}")
    
    print("\n[WEIGHTED METRICS]")
    print(f"  Precision: {metrics['weighted_precision']:.4f}")
    print(f"  Recall:    {metrics['weighted_recall']:.4f}")
    print(f"  F1-Score:  {metrics['weighted_f1']:.4f}")
    
    print("\n[OTHER METRICS]")
    print(f"  Samples F1:        {metrics['samples_f1']:.4f}")
    print(f"  Hamming Loss:      {metrics['hamming_loss']:.4f}")
    print(f"  Exact Match Ratio: {metrics['exact_match_ratio']:.4f}")
    
    print("=" * 60)

In [ ]:
# ============================================
# EVALUATE ON TEST SET
# ============================================

# Load best model
checkpoint = torch.load('best_model.pt')
sentiment_model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")

# Evaluate on test set
test_loss, test_preds, test_labels = validate(
    sentiment_model, test_loader, criterion, device
)

print(f"\nTest Loss: {test_loss:.4f}")

# Compute metrics
test_metrics = compute_metrics(test_preds, test_labels, ID2LABEL)
print_metrics(test_metrics)

In [ ]:
# ============================================
# PER-CLASS METRICS
# ============================================

def compute_per_class_metrics(preds: torch.Tensor, labels: torch.Tensor, id2label: dict):
    """Tinh metrics cho tung class."""
    preds_np = preds.numpy()
    labels_np = labels.numpy()
    
    print("\n" + "=" * 80)
    print("PER-CLASS METRICS")
    print("=" * 80)
    print(f"{'Label':<30} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 80)
    
    results = []
    for idx in range(preds_np.shape[1]):
        label_name = id2label[idx]
        y_true = labels_np[:, idx]
        y_pred = preds_np[:, idx]
        
        support = int(y_true.sum())
        if support == 0:
            precision = recall = f1 = 0.0
        else:
            precision = precision_score(y_true, y_pred, zero_division=0)
            recall = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
        
        results.append({
            'label': label_name,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': support,
        })
        
        print(f"{label_name:<30} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f} {support:>10}")
    
    print("=" * 80)
    return results


# Compute per-class metrics
per_class_results = compute_per_class_metrics(test_preds, test_labels, ID2LABEL)

---
## 8. Inference voi VLM (InternVL3)

Su dung full pipeline: Image -> Preprocessor -> ViT -> Projection -> LLM -> de-Tokenizer -> Output Text

In [ ]:
# ============================================
# INFERENCE FUNCTION
# ============================================

# Prompt templates
SENTIMENT_PROMPT_VI = """Phan tich cam xuc/y kien duoc the hien trong hinh anh nay.
Xem xet ca noi dung hinh anh va bat ky van ban nao xuat hien.

Phan loai cam xuc tong the thanh:
- Tich cuc (positive): The hien su hai long, vui ve, dong y
- Tieu cuc (negative): The hien su khong hai long, tuc gian, phan doi
- Trung lap (neutral): Khach quan, khong co cam xuc ro rang

Tra loi theo dinh dang sau:
Cam xuc: [phan loai]
Giai thich: [giai thich ngan gon]
"""

SENTIMENT_PROMPT_EN = """Analyze the sentiment/opinion expressed in this image.
Consider both the visual content and any text present.

Classify the overall sentiment as:
- Positive: Expressing satisfaction, happiness, approval
- Negative: Expressing dissatisfaction, anger, disapproval
- Neutral: Factual, no clear emotional tone

Provide your analysis in the following format:
Sentiment: [classification]
Reasoning: [brief explanation]
"""


def load_image_for_vlm(image, input_size=448):
    """
    Preprocessor cho VLM inference.
    
    Input: PIL Image
    Output: pixel_values tensor cho model.chat()
    """
    transform = build_transform(input_size=input_size)
    
    if isinstance(image, Image.Image):
        pixel_values = transform(image).unsqueeze(0)
    else:
        pixel_values = transform(Image.open(image)).unsqueeze(0)
    
    pixel_values = pixel_values.to(model.device, dtype=torch_dtype)
    return pixel_values


def analyze_sentiment_vlm(image: Image.Image, language: str = "vi") -> str:
    """
    Phan tich sentiment tu hinh anh su dung VLM.
    
    Pipeline: Image -> Preprocessor -> ViT -> Projection -> LLM -> Output Text
    
    Args:
        image: PIL Image
        language: "vi" hoac "en"
    
    Returns:
        Generated text (sentiment analysis)
    """
    prompt = SENTIMENT_PROMPT_VI if language == "vi" else SENTIMENT_PROMPT_EN
    
    # Preprocessor: Image -> Tensor
    pixel_values = load_image_for_vlm(image)
    
    # Add image token to prompt
    prompt_with_image = f"<image>\n{prompt}"
    
    # LLM generate (ViT -> Projection -> LLM -> de-Tokenizer)
    generation_config = {
        "max_new_tokens": 256,
        "do_sample": False,
    }
    
    response = model.chat(
        tokenizer,
        pixel_values,
        prompt_with_image,
        generation_config
    )
    
    return response

In [ ]:
# Demo voi test image
print("=== Phan tich anh mau voi VLM ===\n")

display(test_image.resize((300, 300)))

result = analyze_sentiment_vlm(test_image, language="vi")
print("\n" + "=" * 50)
print("OUTPUT:")
print("=" * 50)
print(result)
print("=" * 50)

---
## Tong Ket

### Kien truc:

```
                      [ Embeddings ]───( Word-pieces )───>[ de-Tokenizer ]
                            ^                                    |
                            |                                    v
                      ( Lang tokens )                      (( Output Text ))
                            ^
                            |
    +-------------------------------------------------------+
    |                         LLM                           |
    +-------------------------------------------------------+
         ^                               ^
         |                               |
   (Lang tokens)                   (Lang tokens)
         |                               |
    [ Projection ]                [  Embeddings  ]
         ^                               ^
    (Img tokens)                   (word-pieces)
         |                               |
    [    ViT     ]                [  Tokenizer   ]
         ^                               ^
   (14px patches)                        |
         |                               |
    [Preprocessor]                       |
         ^                               |
         |                               |
  ((   Image   ))                 (( Input text  ))
```

### Components:

| Component | Input | Output | Trainable |
|-----------|-------|--------|-----------|
| Preprocessor | Image (any size) | Tensor [3, 224, 224] | - |
| ViT (google/vit-large-patch16-224) | Tensor [3, 224, 224] | Img tokens [197, 1024] | Frozen |
| Projection (MLP) | Img tokens [197, 1024] | Lang tokens [197, 4096] | Trainable |
| Tokenizer | Text | Word-pieces | - |
| Embeddings | Word-pieces | Lang tokens [N, 4096] | Frozen |
| LLM (InternVL3-8B) | Lang tokens | Lang tokens | Frozen |
| de-Tokenizer | Word-pieces | Text | - |

### Dataset: ViMACSA

| Split | Samples |
|-------|---------|
| Train | ~3000 |
| Dev | ~500 |
| Test | ~500 |

### Labels: 3 Sentiments

- Positive
- Negative
- Neutral